In [6]:
import geopandas as gpd
import pandas as pd
import numpy as np
import spreg
from libpysal.weights import Queen
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. CARGAR Y PREPARAR DATOS
# ============================================

ruta_zonas = r"C:/Users/manuz/OneDrive/Maestria/Análisis geoespacial/Proyecto_Curso/Datos/Datos_procesados/Zonas_Medellin.gpkg"
zonas = gpd.read_file(ruta_zonas)

ruta_temp = r"C:/Users/manuz/OneDrive/Maestria/Análisis geoespacial/Proyecto_Curso/Datos/Datos_procesados/Temperatura_por_zona.csv"
df_temp = pd.read_csv(ruta_temp)

temp_promedio = df_temp.groupby('ID').agg({
    'Temp_media': 'mean'
}).reset_index()

zonas_temp = zonas.merge(temp_promedio, left_index=True, right_on='ID', how='left')
zonas_urbano = zonas_temp[zonas_temp['Tipo'] == 'Urbano'].copy()

# Calcular densidad de árboles
ruta_arboles = r"C:/Users/manuz/OneDrive/Maestria/Análisis geoespacial/Proyecto_Curso/Datos/Datos_procesados/Arboles_Activos_Medellin_TODOS.gpkg"
arboles = gpd.read_file(ruta_arboles, columns=['geometry'])

arboles_en_zonas = gpd.sjoin(arboles, zonas_urbano, how='inner', predicate='within')
conteo_arboles = arboles_en_zonas.groupby('ID').size().reset_index(name='num_arboles')

zonas_urbano = zonas_urbano.merge(conteo_arboles, on='ID', how='left')
zonas_urbano['num_arboles'] = zonas_urbano['num_arboles'].fillna(0).astype(int)
zonas_urbano['area_km2'] = zonas_urbano.geometry.area / 1_000_000
zonas_urbano['densidad_arboles'] = zonas_urbano['num_arboles'] / zonas_urbano['area_km2']

# Definir variables
y = zonas_urbano['Temp_media'].values.reshape((-1, 1))
X_vars = ['densidad_arboles', 'area_km2']
X = zonas_urbano[X_vars].values

# Estandarizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Matriz de pesos Queen
w = Queen.from_dataframe(zonas_urbano)
w.transform = 'r'

print("Datos preparados. Barrios:", len(zonas_urbano))

# ============================================
# 2. MODELO SLX (Spatial Lag of X)
# ============================================

wx = zonas_urbano[X_vars].apply(
    lambda col: spreg.weights.spatial_lag.lag_spatial(w, col)
)
wx.columns = [f'w_{col}' for col in X_vars]

slx_exog = np.column_stack([X_scaled, wx.values])
slx_model = spreg.OLS(y, slx_exog,
                      name_y='Temp_media',
                      name_x=X_vars + list(wx.columns),
                      name_w='Queen')

print("\n" + "="*60)
print("MODELO SLX (Spatial Lag of X)")
print("="*60)
print(slx_model.summary)

# ============================================
# 3. MODELO SEM (Spatial Error Model)
# ============================================

sem_model = spreg.GM_Error_Het(y, X_scaled, w=w,
                               name_y='Temp_media',
                               name_x=X_vars,
                               name_w='Queen')

print("\n" + "="*60)
print("MODELO SEM (Spatial Error Model)")
print("="*60)
print(sem_model.summary)

# ============================================
# 4. MODELO SAR-LAG (Spatial Lag Model)
# ============================================

sar_lag_model = spreg.GM_Lag(y, X_scaled, w=w,
                             name_y='Temp_media',
                             name_x=X_vars,
                             name_w='Queen')

print("\n" + "="*60)
print("MODELO SAR-LAG (Spatial Lag Model)")
print("="*60)
print(sar_lag_model.summary)

# ============================================
# 5. MODELO SDM (Spatial Durbin Model)
# ============================================

sdm_exog = np.column_stack([X_scaled, wx.values])
sdm_model = spreg.GM_Lag(y, sdm_exog, w=w,
                         name_y='Temp_media',
                         name_x=X_vars + list(wx.columns),
                         name_w='Queen')

print("\n" + "="*60)
print("MODELO SDM (Spatial Durbin Model)")
print("="*60)
print(sdm_model.summary)

# ============================================
# 6. COMPARACIÓN DE MODELOS
# ============================================

modelos = {
    'SLX': slx_model,
    'SEM': sem_model,
    'SAR-Lag': sar_lag_model,
    'SDM': sdm_model
}

comparacion = []
for nombre, modelo in modelos.items():
    try:
        r2 = modelo.r2 if hasattr(modelo, 'r2') else None
        aic = modelo.aic if hasattr(modelo, 'aic') else None
        log_lik = modelo.llik if hasattr(modelo, 'llik') else None
        comparacion.append({
            'Modelo': nombre,
            'R²': r2,
            'AIC': aic,
            'Log-Lik': log_lik
        })
    except:
        pass

df_comparacion = pd.DataFrame(comparacion)

print("\n" + "="*60)
print("COMPARACIÓN DE MODELOS")
print("="*60)
print(df_comparacion.to_string(index=False))

# Recomendación
if not df_comparacion[df_comparacion['AIC'].notna()].empty:
    mejor_modelo = df_comparacion.loc[df_comparacion['AIC'].idxmin()]
    print(f"\nEl modelo con mejor ajuste (menor AIC) es: {mejor_modelo['Modelo']}")
    print(f"  R²: {mejor_modelo['R²']:.4f}")
    print(f"  AIC: {mejor_modelo['AIC']:.2f}")
else:
    print("\nNo se pudo determinar el mejor modelo por AIC.")

Datos preparados. Barrios: 271

MODELO SLX (Spatial Lag of X)
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :        None
Dependent Variable  :  Temp_media                Number of Observations:         271
Mean dependent var  :     36.5803                Number of Variables   :           5
S.D. dependent var  :      3.3011                Degrees of Freedom    :         266
R-squared           :      0.1634
Adjusted R-squared  :      0.1508
Sum squared residual:      2461.5                F-statistic           :     12.9885
Sigma-square        :       9.254                Prob(F-statistic)     :   1.126e-09
S.E. of regression  :       3.042                Log likelihood        :    -683.501
Sigma-square ML     :       9.083                Akaike info criterion :    1377.001
S.E of regression ML:      3.0138        